# RAG with LangChain

Before we dive into Verbatim RAG, let's first see how a traditional RAG system works. This will help us understand the problems that Verbatim RAG solves.

Traditional RAG systems:
1. **Split documents** into chunks
2. **Embed chunks** into vector space  
3. **Retrieve** relevant chunks for a query
4. **Generate answers** freely based on the retrieved context

The key issue: The LLM can generate plausible-sounding information that wasn't actually in the source documents!

In [ ]:
# Install LangChain for comparison
!pip install "langchain==0.3.27" langchain-openai langchain-community faiss-cpu openai pypdf 

In [ ]:
!pip install rich

In [ ]:
from rich.console import Console

console = Console()

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document as LangChainDocument


# Load PDFs using LangChain's PyPDFLoader (traditional PDF parsing)
console.print("Loading PDFs with PyPDFLoader...")
loader1 = PyPDFLoader("https://aclanthology.org/2025.bionlp-share.8.pdf")
loader2 = PyPDFLoader("https://aclanthology.org/2020.lrec-1.448.pdf")

langchain_docs = []
langchain_docs.extend(loader1.load())
langchain_docs.extend(loader2.load())

console.print(f"Loaded {len(langchain_docs)} document pages for traditional RAG")
console.print(f"\nSample content from PyPDFLoader:")
console.print(f"Page 1 preview: {langchain_docs[0].page_content[:200]}...")

In [ ]:
# Traditional RAG: Sentence-based text splitting for fairer comparison
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],  # tries to split at sentence boundaries first
    chunk_size=1024,
    chunk_overlap=50
)

# Split documents into chunks
splits = text_splitter.split_documents(langchain_docs)
console.print(f"Created {len(splits)} sentence-based chunks")

# Show a sample chunk
console.print("\nSample chunk:")
console.print(f"Content: {splits[0].page_content}...")
console.print(f"Metadata: {splits[0].metadata}")

In [ ]:
import os
# Add your key here
# os.environ["OPENAI_API_KEY"] = 

In [ ]:
# Set up OpenAI API (you need to have OPENAI_API_KEY set)
# Make sure you have: export OPENAI_API_KEY=your_api_key_here

from langchain.chat_models import ChatOpenAI

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(splits, embeddings)

# Create the traditional QA chain with gpt-4o-mini
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=1.0, model_name="gpt-5-mini"),
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    verbose=False,
)

console.print("Traditional RAG system set up successfully!")


In [ ]:
# Query the traditional RAG system
question = "How much synthetic data they generated in ArchEHR-QA 2025?"

result = qa_chain({"query": question})

console.print("## Traditional RAG Answer:")
console.print(result["result"])

console.print(f"\n## Source Documents Used ({len(result['source_documents'])}):")
for i, doc in enumerate(result["source_documents"][:3]):  # Show first 3 sources
    console.print(f"\n**Source {i + 1}:** {doc.metadata.get('title', 'Unknown')}")
    console.print(f"Content preview: {doc.page_content[:150]}...")

## Problems with Traditional RAG

Traditional RAG systems like the one above have several issues:

### 1. **Hallucination Risk**
- The LLM can generate plausible-sounding but incorrect information
- Numbers might be rounded or approximated ("around 58,000" vs exact "58k")
- The model might combine information from multiple sources incorrectly

### 2. **Poor Traceability**  
- Hard to verify exactly where specific claims come from
- Source documents are provided but without precise mappings to claims
- Difficult to fact-check individual statements

### 3. **No Guarantee of Grounding**
- Even with source documents, there's no guarantee the answer only uses information from them
- The LLM might fill in gaps with its training knowledge

Let's try a different approach, Verbatim RAG!

# Verbatim RAG

<p align="center">
  <img src="https://github.com/KRLabsOrg/verbatim-rag/blob/main/assets/chiliground.png?raw=true" alt="ChiliGround Logo" width="400"/>
  <br><em>Chill, I Ground! 🌶 ️</em>
</p>

A minimalistic approach to Retrieval-Augmented Generation (RAG) that prevents hallucination by ensuring all generated content is explicitly derived from source documents.

[![PyPI](https://img.shields.io/pypi/v/verbatim-rag)](https://pypi.org/project/verbatim-rag/)
[![License](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![ACL 2025](https://img.shields.io/badge/ACL%20Anthology-2025.bionlp--share.8-blue)](https://aclanthology.org/2025.bionlp-share.8/)

## Concept

Traditional RAG systems retrieve relevant documents and then allow an LLM to freely generate responses based on that context. This can lead to hallucinations where the model invents facts not present in the source material.

Verbatim RAG solves this by extracting verbatim text spans from documents and composing responses entirely from these exact passages, with direct citations linking back to sources.

For extraction, we can use LLM-based span extractors or fine-tuned encoder-based models like ModernBERT. We've trained our own ModernBERT model for this purpose, which is available on [HuggingFace](https://huggingface.co/KRLabsOrg/verbatim-rag-modern-bert-v1) (we've trained it on the [RAGBench](https://huggingface.co/datasets/galileo-ai/ragbench) dataset).

With this approach, **the whole RAG pipeline can be run without any usage of LLMs**, and with using SPLADE embeddings, the pipeline can be run entirely on CPU, making it lightweight and efficient.


Lets try it out with a few papers and simple examples!

## First steps

In [ ]:
# Install dependencies
!pip install verbatim-rag

In [ ]:
# Define the schema
from verbatim_rag import VerbatimRAG, VerbatimIndex
from verbatim_rag.schema import DocumentSchema

# Add two papers
paper = DocumentSchema.from_url(
    url="https://aclanthology.org/2025.bionlp-share.8.pdf",
    title="KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering",
    doc_type="academic_paper",
    authors=["Adam Kovacs", "Paul Schmitt", "Gabor Recski"],
    conference="BioNLP",
    year=2025,
    category="nlp",
)

paper2 = DocumentSchema.from_url(
    url="https://aclanthology.org/2020.lrec-1.448.pdf",
    title="Better Together: Modern Methods Plus Traditional Thinking in NP Alignment",
    doc_type="academic_paper",
    authors=["Adam Kovacs", "Judit Acs", "Andras Kornai", "Gabor Recski"],
    conference="LREC",
    year=2020,
    category="nlp",
)

In [ ]:
# Papers are parsed into markdown
console.print(paper.content[:500])

In [ ]:
# If you want, you can define your content manually
paper_manual = DocumentSchema(
    content="""
    # Title
    ## Authors
    Adam Kovacs, Paul Schmitt, Gabor Recski
    """,
    title="KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering",
    doc_type="academic_paper",
    authors=["Adam Kovacs", "Paul Schmitt", "Gabor Recski"],
    conference="BioNLP",
    year=2025,
    category="nlp",
)

## Chunking

Chunking is the process of splitting the content of a document into smaller, more manageable chunks. This is important because it allows us to retrieve the most relevant information from the document.

While in theory, chunking is a simple process, in practice it can be quite complex, but its always worth it to do it right.

In [ ]:
from verbatim_rag.chunker_providers import MarkdownChunkerProvider


chunker = MarkdownChunkerProvider(
     min_chunk_size=500,
     max_chunk_size=5000,
)

chunks = chunker.chunk(paper.content)

console.print(f"Chunk 1: {chunks[1][0]}")
console.print(f"Chunk 2: {chunks[2][0]}")

Many document types (e.g. markdown) have a certain structure, that is something we shouldn't lose with the chunking process.

In [ ]:
console.print(f"Chunk 1: {chunks[1][1]}")
console.print(f"Chunk 2: {chunks[2][1]}")

## Indexing

We can now index the chunks. This is the process of converting the chunks into a vector space, so we can use them for retrieval. Usually very resource intensive part if you have a lot of documents.

In [ ]:
from verbatim_rag.embedding_providers import SentenceTransformersProvider
from verbatim_rag.vector_stores import LocalMilvusStore

dense_provider = SentenceTransformersProvider(
    model_name="ibm-granite/granite-embedding-english-r2", device='cpu'
)
vector_store = LocalMilvusStore(
    db_path="./rag_lecture.db",
    collection_name='rag_lecture',
    dense_dim=dense_provider.get_dimension(),
    enable_dense=True,
    enable_sparse=False,
    nlist=16384,
)
index = VerbatimIndex(
        vector_store=vector_store,
        dense_provider=dense_provider,
        chunker_provider=chunker,
    )

index.add_documents([paper, paper2])

In [ ]:
chunks = index.get_all_chunks()

# First chunk
console.print(f"Chunk 1: {chunks[0]}")

In [ ]:
# You can filter by metadata, very useful for document type, conference, year, user_id, etc.
index.query(
    filter="metadata['title'] == 'KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering'",
    k=1,
)

## Lets do some RAG

Now that we have our index, we can use it to answer questions.

We'll use the `VerbatimRAG` class to answer questions.

In [ ]:
from verbatim_rag.core import LLMClient

llm_client = LLMClient(model="gpt-5.1", temperature=1.0)

rag = VerbatimRAG(index, llm_client=llm_client)

response = rag.query("How much synthetic data they generated in Kovacs et al. 2025?")

In [ ]:
console.print(response.answer)

In [ ]:
console.print(response.structured_answer)

By default, **VerbatimRAG** uses a dynamic template generation strategy, generating a template for each question with LLMs to ensure the best possible answer.

A special mode is also available, where you want full control over your answer, that case you can use the `static` mode, where you can provide the exact template you want your answer to be in.

Try it out:

In [ ]:
template = """
Thanks for your question!

You can find the answer to your question in the following sources:
[RELEVANT_SENTENCES]
"""


rag.template_manager.use_static_mode(template)

response = rag.query("How much synthetic data they generated in Kovacs et al. 2025?")

console.print(response.answer)